In [0]:
from pyspark.sql.functions import *

# =========================
# CONFIG
# =========================
catalog_name = "data_dev_olist"
schema_name = "silver"
table_name = "clean_customer"

source_path = (
    "abfss://raw-data@quocluudata.dfs.core.windows.net/"
    "bronze_delta/olist_customers"
)

silver_path = (
    "abfss://raw-data@quocluudata.dfs.core.windows.net/"
    "silver/clean_customer"
)

target_table = f"{catalog_name}.{schema_name}.{table_name}"

# =========================
# READ BRONZE
# =========================
df = (
    spark.read
    .format("delta")
    .load(source_path)
)

# =========================
# TRANSFORM
# =========================
processed_df = (
    df
    .na.drop()
    .dropDuplicates(["customer_id"])
    .withColumn("is_active", lit(True))
    .withColumn("_processed_at", current_timestamp())
)

# =========================
# WRITE DELTA TO ADLS
# =========================
(
    processed_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_path)
)

# =========================
# CREATE EXTERNAL TABLE
# =========================
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {target_table}
USING DELTA
LOCATION '{silver_path}'
""")

print(f"Saved {processed_df.count()} rows")
print(f"External Table : {target_table}")
print(f"Location       : {silver_path}")

Saved 99441 rows
External Table : data_dev_olist.silver.clean_customer
Location       : abfss://raw-data@quocluudata.dfs.core.windows.net/silver/clean_customer


In [0]:
%sql
 table data_dev_olist.silver.clean_customer;

In [0]:
%sql
DESCRIBE EXTENDED data_dev_olist.silver.clean_customer;


col_name,data_type,comment
customer_id,string,null
customer_unique_id,string,null
customer_zip_code_prefix,int,null
customer_city,string,null
customer_state,string,null
is_active,boolean,null
_processed_at,timestamp,null
,,
# Delta Statistics Columns,,
Column Names,"is_active, customer_id, customer_state, customer_zip_code_prefix, _processed_at, customer_unique_id, customer_city",


In [0]:
display(
    dbutils.fs.ls(
        "abfss://raw-data@quocluudata.dfs.core.windows.net/silver/clean_customer"
    )
)

path,name,size,modificationTime
abfss://raw-data@quocluudata.dfs.core.windows.net/silver/clean_customer/_delta_log/,_delta_log/,0,1781286465000
abfss://raw-data@quocluudata.dfs.core.windows.net/silver/clean_customer/part-00000-9333193a-4b3d-49d3-86cb-0300e09de83f.c000.snappy.parquet,part-00000-9333193a-4b3d-49d3-86cb-0300e09de83f.c000.snappy.parquet,6917550,1781286466000


In [0]:
%sql 
DESCRIBE DETAIL data_dev_olist.silver.clean_customer;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,353e0f99-9a5d-457d-8a28-246fd169f7d2,data_dev_olist.silver.clean_customer,null,abfss://raw-data@quocluudata.dfs.core.windows.net/silver/clean_customer,2026-06-12T17:47:45.507Z,2026-06-12T17:47:46.000Z,List(),List(),1,6917550,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql 
SHOW CREATE TABLE data_dev_olist.silver.clean_customer;

createtab_stmt
"CREATE TABLE data_dev_olist.silver.clean_customer ( customer_id STRING COLLATE UTF8_BINARY, customer_unique_id STRING COLLATE UTF8_BINARY, customer_zip_code_prefix INT, customer_city STRING COLLATE UTF8_BINARY, customer_state STRING COLLATE UTF8_BINARY, is_active BOOLEAN, _processed_at TIMESTAMP) USING delta LOCATION 'abfss://raw-data@quocluudata.dfs.core.windows.net/silver/clean_customer' TBLPROPERTIES ( 'delta.enableDeletionVectors' = 'true', 'delta.feature.appendOnly' = 'supported', 'delta.feature.deletionVectors' = 'supported', 'delta.feature.invariants' = 'supported', 'delta.minReaderVersion' = '3', 'delta.minWriterVersion' = '7')"


In [0]:
%sql
SELECT * 
FROM delta.`abfss://raw-data@quocluudata.dfs.core.windows.net/silver/clean_customer`
LIMIT 10;

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,is_active,_processed_at
237098a64674ae89babdc426746260fc,4390ddbb6276a66ff1736a6710205dca,82820,curitiba,PR,true,2026-06-12T17:47:45.680Z
e3109970a3fe8021d5ff82c577ce5606,a8654e2af5da6bb72f52c22b164855e1,5528,sao paulo,SP,true,2026-06-12T17:47:45.680Z
c532a74a3ebf1bacce2e2bcce3783317,91ec50a00ae74d0a229d2efdf4344e1e,14026,ribeirao preto,SP,true,2026-06-12T17:47:45.680Z
19cecb194f54e614b70d971306a9931b,d251c190ca75786e9ab937982d60d1d4,30320,belo horizonte,MG,true,2026-06-12T17:47:45.680Z
c82a5e4fafdbeb34f08928ccfba27d14,ca19a17e381182923b66007a351574b7,85854,foz do iguacu,PR,true,2026-06-12T17:47:45.680Z
b06429ef920fcfdd75713c712c9ee7b7,9316f45a5da8403a5938bd6069b1a4a7,12240,sao jose dos campos,SP,true,2026-06-12T17:47:45.680Z
d3ab15f0bd2c58865d566ab645572cd5,9ccfff93c79f3dd996cce15f26480c5b,21615,rio de janeiro,RJ,true,2026-06-12T17:47:45.680Z
031cd5f826be3d804771e3e3a1b21a1c,717aa48025662fcf27ddebbecc5f782b,41706,salvador,BA,true,2026-06-12T17:47:45.680Z
79adcf02229a33e78f0f5412a2434f53,8c8fccc50566baaed602e3775d9d5665,21515,rio de janeiro,RJ,true,2026-06-12T17:47:45.680Z
e3c7e245a96d7fa339fe6c16f8da4e90,79051ee5ee98c4bd6982e67e2e79dbcb,7847,franco da rocha,SP,true,2026-06-12T17:47:45.680Z
